# 4 · Discover, download, and add a layer to RustGIS

This standalone tutorial discovers one Sentinel-2 Item through Earth Search, selects one conservative public HTTPS data-role COG candidate, and renders explicit actions for the provider file and [RustGIS Web](https://web.rustgis.app/). Run all cells in JupyterLite. Metadata requests begin when you run the connection/search cell; raster bytes are **not** requested until you choose the Download or RustGIS action.

Public metadata does not guarantee asset access. The provider still controls CORS, HTTP Range support, file size, licensing, and availability. If discovery fails because of CORS, TLS, HTTP status, timeout, or provider outage, the client surfaces that error. Retry or choose another catalog; the notebook never fabricates results.

In [ ]:
from stac_browser import STACBrowser, PRESETS

client = await STACBrowser(PRESETS['earthsearch']).connect()
items = await client.search(
    bbox=[-123.1, 37.2, -121.8, 38.1],
    collection='sentinel-2-l2a',
    start='2024-06-01', end='2024-06-15', limit=1)
if not items:
    raise RuntimeError('No Items matched. Retry later or broaden the bounded dates/area.')
item = items[0]
print('Discovered Item:', item['id'])
print('Collection:', item.get('collection'))
print('Available assets:', list(item.get('assets', {})))

## Select one exact public COG candidate

The helper is deliberately narrower than the full Explorer. It accepts only a loaded Item's primary asset when it is HTTPS, data-role, TIFF-declared, named `.tif`, `.tiff`, or `.cog`, and free of credentials, signed query parameters, `auth:refs`, supporting roles, and the protected Copernicus download origin. A suffix is only a candidate signal; RustGIS validates the actual bytes.

In [ ]:
handoff = client.public_cog_handoff(item)
selected_asset = item['assets'][handoff['asset_key']]
collection_metadata = next((entry for entry in client.collections if entry.get('id') == item.get('collection')), {})
print('Selected candidate asset:', handoff['asset_key'])
print('Asset title:', selected_asset.get('title'))
print('Asset media type:', selected_asset.get('type'))
print('Asset roles:', selected_asset.get('roles'))
print('Collection license:', collection_metadata.get('license', 'not advertised'))
print('Collection providers:', collection_metadata.get('providers', []))
print('Exact provider address:', handoff['href'])
print('No raster request has been made by this helper.')

## User-activated actions

**Download selected GeoTIFF/COG candidate** navigates directly to the provider. It may open in a tab; use the browser's Save command when needed. **Add exact candidate to RustGIS** opens a blank RustGIS project with the exact selected address in its documented `data` parameter. A TIFF declaration and filename suffix do not prove valid COG bytes; RustGIS performs its own validation. Neither action proxies or signs the asset. A destination error means the provider or format did not satisfy RustGIS's browser requirements.

In [ ]:
import json
from pathlib import Path
from html import escape
from IPython.display import HTML, display

display(HTML(
    '<p><a href="' + escape(handoff['download'], quote=True) + '" target="_blank" rel="noopener noreferrer" download>'
    'Download selected GeoTIFF/COG candidate</a></p>'
    '<p><a href="' + escape(handoff['rustgis'], quote=True) + '" target="_blank" rel="noopener noreferrer">'
    'Add exact candidate to RustGIS ↗</a></p>'))
client.save('rustgis-tutorial')
for saved_name in ('rustgis-tutorial-items.geojson', 'rustgis-tutorial-search.json'):
    json.loads(Path(saved_name).read_text(encoding='utf-8'))
print('Saved rustgis-tutorial-items.geojson and rustgis-tutorial-search.json')
print('RustGIS workflow ready')

## Attribution and broader formats

Inspect the Item, collection license, and provider attribution before using or publishing the layer. The full Explorer supports more RustGIS formats, safe public alternates, and STAC Item handoffs; use it when this intentionally narrow COG tutorial reports no candidate. Download notebook-created JSON files from JupyterLite's file browser if you want durable copies.